# Module 8 Assignment: Give Your Persona Agent Memory With Sessions

Lab 7 added a `SQLiteSession` to Study Buddy: first in memory, to fix two `Runner.run` calls that
did not remember each other, then file backed, so the conversation survives a kernel restart. The
Gradio `chat()` function got simpler in the process, no more rebuilding `history` by hand.

This assignment makes you do the same thing to your **own** Persona Agent, the one you rebuilt on
the OpenAI Agents SDK in the Module 7 assignment. Nothing about your agent's persona, notes, or
three tools needs to change here: you are giving it memory, not redesigning it.

**What you'll build:**
1. Your Module 7 assignment's Agent, Azure configuration, and three tools, carried over unchanged
2. A quick demonstration that two separate `Runner.run` calls do not remember each other
3. A file backed `SQLiteSession` that fixes it
4. A simplified Gradio `chat()` function that uses `session=session` instead of rebuilding `history`

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you get stuck on any step, look at how Lab 7 made the same kind of change to Study Buddy, and do
the analogous thing to your own agent.

## 0. Install the packages (run once)

Same as Lab 7: no new package beyond what your Module 7 assignment already installed.
`SQLiteSession` ships with `openai-agents` itself.

In [ ]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

## 1. Imports

**TODO:** Copy the imports from your Module 7 assignment, plus one new name: `SQLiteSession`,
imported from `agents` alongside `Agent` and `Runner`, the same way Lab 7, Step 1 added it.

In [ ]:
from dotenv import load_dotenv
import os
import requests
from urllib.parse import quote
from openai import AsyncAzureOpenAI
from agents import (
    Agent,
    Runner,
    SQLiteSession,
    function_tool,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
)
from pypdf import PdfReader
import gradio as gr


## 2. Load your Azure OpenAI credentials from `.env`

Identical to the Module 7 assignment. Sessions do not change which provider you talk to.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

## 3. Configure the OpenAI Agents SDK for Azure OpenAI

**TODO:** Copy this setup from your Module 7 assignment, unchanged: `set_tracing_disabled(True)`,
your `AsyncAzureOpenAI` client, and `OpenAIChatCompletionsModel`. A session is a feature of
`Runner.run`, not of the model or the client, so nothing here needs to change.

In [ ]:
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)


## 4. Your persona, notes, and three tools

**TODO:** Bring over your persona, your notes, and all three of your `@function_tool` decorated
tools from the Module 7 assignment, unchanged. None of them need to change for this assignment: a
session only affects how conversation history is stored and replayed, not what a tool does.

In [ ]:
persona = (
    "You are Professor Cell, a friendly and helpful biology tutor "
    "who helps students understand cell theory, cell types, and organelles. "
    "You use simple explanations, give encouraging feedback, and clearly "
    "explain mistakes without making students feel embarrassed."
)

reader = PdfReader("notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

print(persona)
print(notes[:500], "...")


In [ ]:
mistakes = []


@function_tool
def record_biology_mistake(question: str) -> str:
    """Record a biology question the student got wrong or answered with idk.

    Args:
        question: The study question the student missed.
    """
    question = (question or "").strip()
    if not question:
        return "No question given, so nothing was recorded."

    mistakes.append(question)
    print(f"[mistake] {question}")
    return f"Recorded. {len(mistakes)} question(s) to review so far."


In [ ]:
@function_tool
def api_tool(value: str) -> str:
    """Look up a summary of a biology definition for example cell type.

    Args:
        value: The biology definition to look up for example mitochondria.
    """
    definition = (value or "").strip()
    if not definition:
        return "Tell me which biology definition to look up."

    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{quote(definition)}"
    try:
        response = requests.get(
            url,
            headers={"User-Agent": "ProfessorCellTutor/1.0 (biology study assistant)"},
            timeout=10,
        )
    except requests.exceptions.RequestException:
        return "I couldn't reach the website. Try again."

    if response.status_code != 200:
        return f"I couldn't find a summary for '{definition}'. Try another biology definition, like mitochondria."

    data = response.json()
    summary = data.get("extract")
    if not summary:
        return f"I found a page for '{definition}', but it didn't include a summary."

    title = data.get("title", definition)
    return f"{title}: {summary}"


In [ ]:
score = {"right": 0, "total": 0}


@function_tool
def third_tool(value: str) -> str:
    """Update the student's quiz score after each answer and return the score so far.

    Args:
        value: Whether the student's latest answer was "right" or "wrong".
    """
    result = (value or "").strip().lower()
    if result not in ("right", "wrong"):
        return "Score not updated. Use 'right' or 'wrong'."

    score["total"] += 1
    if result == "right":
        score["right"] += 1

    print(f"[score] {score['right']} / {score['total']}")
    return f"Score so far: {score['right']} out of {score['total']} right."


## 5. Build your Agent

**TODO:** Copy this from your Module 7 assignment, unchanged: `name`, `instructions`, your `tools`
list, and `model=azure_model`.

In [ ]:
instructions = f"""
# Your role

{persona}

You quiz a student on the biology study notes below.

# Context

Here are the study notes you are quizzing the student on:

{notes}

# Rules

Ask exactly one question at a time based only on the notes.
Wait for the student's answer before asking the next question.

After the student answers, clearly state whether the answer is
right or wrong. Give a one sentence explanation either way,
staying in character the whole time.

After every answer, use the third_tool tool with "right" or
"wrong" to update the student's score, and tell the
student their score so far.

If the student's answer is wrong, or they say "idk",
use the record_biology_mistake tool to record the missed question
before moving on to the next question.

If the student asks what a biology term from the notes means, or wants
more detail about one, use the api_tool tool to look it up, then explain
it simply in your own words before continuing the quiz.

Only ask about information covered in the notes.
If the student asks about something outside the notes,
explain that it is not covered in this study set.
"""

professor_cell = Agent(
    name="Professor Cell",
    instructions=instructions,
    tools=[record_biology_mistake, api_tool, third_tool],
    model=azure_model,
)


## 6. Confirm the problem: two calls that do not know each other

**TODO:** Run your agent twice in a row with `await Runner.run(your_agent, "...")`, the same way
Lab 7, Step 7 did with Study Buddy. Pick a first message that gives the agent some information (for
example, ask it a question whose answer it needs to remember), then a second message that only
makes sense if the agent remembers the first (for example, "what did I just ask you?" or "say that
again"). Confirm the second reply shows the agent has no idea what you mean.

In [18]:
result = await Runner.run(
    professor_cell,
    "Please help  me with understanding mitochondria.",
)
print(result.final_output)


Of course! Mitochondria are organelles that make ATP, which is the cell’s main usable energy source. That’s why they’re often called the “powerhouse of the cell.”

In simple terms:
- food is broken down
- mitochondria use that energy in cellular respiration
- the cell gets ATP to power its activities

A few key points:
- Mitochondria are found in eukaryotic cells.
- Their main job is energy production.
- Cells that need lots of energy often have many mitochondria.

Ready for a quiz question? What is the main function of mitochondria?


In [19]:
result = await Runner.run(
    professor_cell,
    "What question did I ask you",
)
print(result.final_output)


You asked, “What question did I ask you?”

Professor Cell here: we haven’t started the quiz yet, so my first question is:

What are the three parts of cell theory?


## 7. Fix it with a file backed `SQLiteSession`

**TODO:** Create a `SQLiteSession`, the same way Lab 7, Step 9 did: give it a `session_id` string,
and a `db_path` so the conversation is written to a file and survives a kernel restart. Then repeat
the same two messages from Step 6, this time passing `session=your_session` to both calls, and
confirm the second reply now correctly remembers the first.

In [20]:
# TODO: your_session = SQLiteSession("your_session_id", "your_conversation_history.db")
professor_session = SQLiteSession("professor_cell_student_1", "professor_cell_history.db")

In [21]:
# TODO: result = await Runner.run(your_agent, "...", session=your_session)
# TODO: print(result.final_output)
result = await Runner.run(
    professor_cell,
    "Please help me with understanding mitochondria.",
    session=professor_session,
)
print(result.final_output)

Of course! Mitochondria are organelles that make ATP, which is the cell’s main energy source. That’s why they’re often called the “powerhouse of the cell.”

In simple terms:
- mitochondria take energy from food
- they use cellular respiration to turn that energy into ATP
- the cell uses ATP to do its jobs

They are found in eukaryotic cells, like plant and animal cells.

Ready to keep going with the quiz?  
Question: What is the main job of mitochondria?


In [23]:
# TODO: result = await Runner.run(your_agent, "...same follow up as Step 6...", session=your_session)
# TODO: print(result.final_output)
result = await Runner.run(
    professor_cell,
    "What question did I just ask you?",
    session=professor_session,
)
print(result.final_output)

You just asked, “What question did I just ask you?”


## 8. Give it a real chat UI with Gradio

**TODO:** Copy this skeleton from Lab 7, Step 11 exactly, structure and all. Notice what is
different from your Module 7 assignment's version: `history` is no longer rebuilt inside `chat()`
at all, `session=your_session` already gives `Runner.run` everything it needs to remember the
conversation.

Try a conversation that would trigger more than one of your three tools across a few turns, then
interrupt the cell (stop button) when you are done chatting.

In [24]:
async def chat(message, history):
    result = await Runner.run(professor_cell, message, session=professor_session)
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Reflection Questions

Answer these in a new Markdown cell at the end of your notebook:

1. In Step 6, what exactly did your agent get wrong on the second call, and why? Point to the
   specific piece of information it had no way of knowing.
   What question did I just ask you? it just repeated the same response basically.
2. Compare your Gradio `chat()` function here to the one in your Module 7 assignment. What code did
   you get to delete, and what does `session=your_session` do in its place?
3. Your `your_session` was created with a `db_path`. If you had left `db_path` out entirely, what
   would still work the same, and what would break the next time you restarted the kernel?
4. `SQLiteSession` also has a `pop_item()` method, demonstrated in Lab 7, Step 10, for undoing the
   last turn. Describe one realistic situation in your own agent's domain where you would want to
   call it.

## Submission Checklist

- Remove your `.env` file before submitting; it contains your API key.
- Remove any `.db` file your session created; it may contain real conversation content.
- Run the notebook top to bottom and confirm it executes cleanly from a fresh kernel.
- Confirm Step 6 shows the agent failing to remember, and Step 7 shows the same scenario working
  correctly once `session=your_session` is added.
- Answer all four reflection questions in a Markdown cell at the end of the notebook.
- Rename the file to include your name, for example `Module8Assignment_JaneDoe.ipynb`, then submit
  it.